In [9]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

TAIEX & Futures

In [ ]:
# Load CSV file & basic data preprocessing
futures_data = pd.read_csv('data/orginal_futures_data.csv')
futures_data['timestamp'] = pd.to_datetime(futures_data['date'] + ' ' + futures_data['time'])
futures_data.set_index('timestamp', inplace=True)

# Remove non-trading hours
futures_data = futures_data.drop(futures_data.between_time('5:00', '8:44').index.union(futures_data.between_time('13:45', '14:59').index))
futures_data = futures_data.fillna(method='ffill')

# Remove NaN values
futures_data = futures_data.dropna()
futures_data = futures_data.drop(columns=['date', 'time', 'contract'])

In [ ]:
# Merge futures and TAIEX data, fill missing values, remove duplicates, and rename columns
# Load TAIEX_data file
TAIEX_data = pd.read_csv('data\orginal_TAIEX_data.csv')
TAIEX_data['timestamp'] = pd.to_datetime(TAIEX_data['date'] + ' ' + TAIEX_data['time'])
TAIEX_data.set_index('timestamp', inplace=True)

# Remove data outside of trading hours
TAIEX_data = TAIEX_data.drop(TAIEX_data.between_time('0:00', '8:59').index.union(TAIEX_data.between_time('13:30', '23:59').index))

# Forward fill missing values
TAIEX_data = TAIEX_data.fillna(method='ffill')

# Remove NaN values
total_TAIEX_data = TAIEX_data.dropna()

# Drop unnecessary columns
total_TAIEX_data = total_TAIEX_data.drop(columns=['date', 'time'])

In [12]:
TX_data = pd.merge(total_TAIEX_data,futures_data, on='timestamp', how='outer')

In [ ]:
# Merge TAIEX and futures data
TX_data = pd.merge(total_TAIEX_data,futures_data, on='timestamp', how='outer').rename(columns={
    'timestamp': 'timestamp',
    'open_x': 'T_open',
    'high_x': 'T_high',
    'low_x': 'T_low',
    'close_x': 'T_close',
    'open_y': 'F_open',
    'high_y': 'F_high',
    'low_y': 'F_low',
    'close_y': 'F_close',
    'vol': 'F_vol'
})

In [ ]:
start_date = '2019-01-01'
TX_data = TX_data.loc[TX_data.index >= start_date]

end_date = '2025-01-01 00:00:00'
TX_data = TX_data.loc[TX_data.index < end_date]

TX_data.fillna(0, inplace=True)

def calculate_TX(row, taiex_initial, future_initial):
    TAIEX_current = row['T_close']
    future_current = row['F_close']
    
    if TAIEX_current != 0:
        TX = TAIEX_current
    else:
        TX = taiex_initial * (1 + (future_current - future_initial) / future_initial)
    
    return round(TX, 2)

# initial
initial_TAIEX = 9685.54
initial_future = 9673

# apply
TX_data['TX'] = TX_data.apply(calculate_TX, args=(initial_TAIEX, initial_future), axis=1)

TX_data.to_csv("data/TX_data.csv", encoding="utf_8_sig")